<a href="https://colab.research.google.com/github/maewhyxng2-prog/zyrquen-platform/blob/main/tools/notebooks/step6000_transfer_validation_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZYRQUEN Native V2 — Step 6000 Candidate vs Step 5000 Baseline Evaluation (`transfer-validation`)

**สถานะเริ่มต้น:** `CANDIDATE — NOT PROMOTED`  
**วัตถุประสงค์:** โหลด Checkpoint Step 6000 (`v2_teacher_transfer_run5000_step6000.json.gz`) และ Baseline Step 5000 (`v2_teacher_transfer_step5000.json.gz`) ใน Runtime ใหม่, ตรวจสอบ SHA-256 ตาม Manifest ของ PR #56 และประเมินผลบนชุดข้อมูล `transfer-validation` เพื่อแสดงตารางเปรียบเทียบ metrics ก่อนพิจารณาเลื่อนสถานะโมเดล

**นโยบายการประเมิน:**
- ใช้โมเดล Native `PureNativeTransformerLMV2` และ `PureContinualTrainerV2` เท่านั้น (ไม่มี external pretrained weights / ไม่มี external AI API)
- ห้ามสร้างตัวเลข metrics จำลอง หากยังไม่กำหนดไฟล์ `VAL_FILE` จะหยุดพร้อมคำแนะนำทันที
- ต้องตรวจ SHA-256 ตรงกับ PR #56 manifest ก่อนโหลดทั้งสอง Checkpoint เสมอ

In [ ]:
#@title 1. ตั้งค่าสภาพแวดล้อมและนำเข้า ZYRQUEN V2 Runtime
import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List

REPO_DIR = Path("/content/zyrquen-platform")
OUT_DIR = Path("/content/zyrquen-resume/checkpoints")
REPO_URL = "https://github.com/maewhyxng2-prog/zyrquen-platform.git"

if not REPO_DIR.exists():
    if (Path.cwd() / "python_core" / "v2" / "zyrquen_v2_pure.py").exists():
        REPO_DIR = Path.cwd()
    elif (Path.cwd().parent.parent / "python_core" / "v2" / "zyrquen_v2_pure.py").exists():
        REPO_DIR = Path.cwd().parent.parent
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

PYTHON_CORE = REPO_DIR / "python_core"
for p in (str(PYTHON_CORE), str(REPO_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

from tools.eval_step6000_vs_step5000 import compare_step6000_vs_step5000
from tools.resume_training import PR56_STEP_MANIFEST, load_texts, resume_from_checkpoint, verify_checkpoint_hash
from v2.zyrquen_v2_pure import PureContinualTrainerV2
from v2.zyrquen_v2_trainer import ByteTokenizer

print(f"REPO_DIR : {REPO_DIR}")
print(f"OUT_DIR  : {OUT_DIR} (exists={OUT_DIR.exists()})")

In [ ]:
#@title 2. ตรวจสอบ SHA-256 (PR #56) และโหลด Step 5000 Baseline + Step 6000 Candidate
baseline_path, baseline_sha = verify_checkpoint_hash(5000, out_dir=OUT_DIR)
candidate_path, candidate_sha = verify_checkpoint_hash(6000, out_dir=OUT_DIR)

baseline_state = resume_from_checkpoint(step=5000, out_dir=OUT_DIR)
candidate_state = resume_from_checkpoint(step=6000, out_dir=OUT_DIR)

print("=== PR #56 Integrity & Fresh-Runtime Reload Verified ===")
for label, info in [("Baseline  (Step 5000)", baseline_state), ("Candidate (Step 6000)", candidate_state)]:
    print(
        f"{label} | file={info['checkpoint_path'].name} | "
        f"step={info['global_step']} | sha256={info['sha256'][:16]}... | "
        f"optimizer={info['optimizer_restored']} | replay={info['replay_buffer_len']}"
    )

In [ ]:
#@title 3. กำหนดชุดข้อมูล Held-Out Transfer-Validation (4 ตัวอย่างแยกขาดจาก Train)
# ระบุ path ของไฟล์ transfer-validation (.jsonl หรือ .json) ที่แยกจากชุดฝึก 16 ตัวอย่าง
VAL_FILE = None  # ตัวอย่าง: Path("/content/zyrquen-resume/data/transfer_validation.jsonl")

transfer_val_texts: List[str] = load_texts(Path(VAL_FILE)) if VAL_FILE else []
if not transfer_val_texts:
    print("[WAIT] ยังไม่ได้กำหนด VAL_FILE — กรุณาระบุไฟล์ transfer-validation ของการทดลองก่อนรัน Cell 4")
else:
    print(f"Loaded {len(transfer_val_texts)} held-out transfer-validation examples from {VAL_FILE}")

In [ ]:
#@title 4. ฟังก์ชันรันการประเมินและแสดงตารางเปรียบเทียบ Metrics (Step 6000 vs Step 5000)
def render_metrics_comparison_table(report: Dict[str, Any]) -> str:
    """สร้างและแสดงตารางเปรียบเทียบ metrics สำหรับ Step 5000 Baseline vs Step 6000 Candidate"""
    bm = report["baseline"]["metrics"]
    cm = report["candidate"]["metrics"]
    d = report["delta"]

    def _pct_delta(base_v: float, cand_v: float) -> str:
        if base_v == 0:
            return "N/A"
        return f"{((cand_v - base_v) / base_v) * 100.0:+.2f}%"

    lines = [
        "==================================================================================",
        "   ZYRQUEN Native V2 — Held-Out Transfer-Validation Metrics Comparison Table      ",
        "==================================================================================",
        f"Validation Examples : {report['validation_examples']}",
        f"Baseline  (Step 5000): {report['baseline']['file']} | sha256={report['baseline']['sha256'][:16]}...",
        f"Candidate (Step 6000): {report['candidate']['file']} | sha256={report['candidate']['sha256'][:16]}...",
        "----------------------------------------------------------------------------------",
        f"{'Metric':<18} {'Step 5000':>14} {'Step 6000':>14} {'Delta':>14} {'Rel %':>12} {'Direction':>10}",
        "----------------------------------------------------------------------------------",
        f"{'loss':<18} {bm['loss']:>14.4f} {cm['loss']:>14.4f} {d['loss']:>+14.4f} {_pct_delta(bm['loss'], cm['loss']):>12} {'lower=better':>10}",
        f"{'perplexity':<18} {bm['perplexity']:>14.3f} {cm['perplexity']:>14.3f} {d['perplexity']:>+14.3f} {_pct_delta(bm['perplexity'], cm['perplexity']):>12} {'lower=better':>10}",
        f"{'accuracy':<18} {bm['accuracy']:>14.4f} {cm['accuracy']:>14.4f} {d['accuracy']:>+14.4f} {_pct_delta(bm['accuracy'], cm['accuracy']):>12} {'higher=better':>10}",
        f"{'tokens':<18} {bm['tokens']:>14d} {cm['tokens']:>14d} {cm['tokens'] - bm['tokens']:>+14d} {'—':>12} {'match=exact':>10}",
        "----------------------------------------------------------------------------------",
        f"Zero Regression on Transfer-Val : {report['zero_regression_on_transfer_val']}",
        f"Promotion Recommendation        : {report['promotion_note']}",
        "==================================================================================",
    ]
    table_str = "\n".join(lines)
    print(table_str)

    try:
        from IPython.display import Markdown, display  # type: ignore

        md = "\n".join([
            "### Transfer-Validation Comparison (Step 5000 vs Step 6000)",
            "| Metric | Step 5000 Baseline | Step 6000 Candidate | Delta | Relative % |",
            "|---|---:|---:|---:|---:|",
            f"| **loss** (lower is better) | {bm['loss']:.4f} | {cm['loss']:.4f} | {d['loss']:+.4f} | {_pct_delta(bm['loss'], cm['loss'])} |",
            f"| **perplexity** (lower is better) | {bm['perplexity']:.3f} | {cm['perplexity']:.3f} | {d['perplexity']:+.3f} | {_pct_delta(bm['perplexity'], cm['perplexity'])} |",
            f"| **accuracy** (higher is better) | {bm['accuracy']:.4f} | {cm['accuracy']:.4f} | {d['accuracy']:+.4f} | {_pct_delta(bm['accuracy'], cm['accuracy'])} |",
            f"| **tokens** | {bm['tokens']} | {cm['tokens']} | {cm['tokens'] - bm['tokens']:+d} | — |",
        ])
        display(Markdown(md))
    except Exception:
        pass
    return table_str


evaluation_report = None
if not transfer_val_texts:
    print("[SKIP] ไม่มีข้อมูล transfer_val_texts จึงยังไม่คำนวณผลเพื่อป้องกันการสร้าง metrics จำลอง")
else:
    evaluation_report = compare_step6000_vs_step5000(out_dir=OUT_DIR, val_texts=transfer_val_texts)
    _ = render_metrics_comparison_table(evaluation_report)

In [ ]:
#@title 5. บันทึกรายงานหลักฐาน (Evidence Artifact) ก่อนพิจารณา Promotion
if evaluation_report is None:
    print("[INFO] ยังไม่มี evaluation_report ให้บันทึก — รัน Cell 3 และ Cell 4 ก่อน")
else:
    evaluation_report["recorded_at_utc"] = datetime.now(timezone.utc).isoformat()
    artifact_path = OUT_DIR / "step6000_vs_step5000_transfer_val_report.json"
    artifact_path.write_text(json.dumps(evaluation_report, indent=2), encoding="utf-8")
    print(f"Saved evidence report: {artifact_path}")
    print(f"Candidate promotion gate passed on transfer-val: {evaluation_report['zero_regression_on_transfer_val']}")